# Crash Course — 연구 설계와 통계 응급 키트

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [crash-course/study-design-stats.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/crash-course/study-design-stats.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 최소 실행 예시

### 예시 1 — Welch t-test + Cohen's d + BH 보정

군당 12명, feature 200개인 가상 데이터. 앞의 10개 feature에만 실제 군 차이를 넣었다. feature마다 평균 차와 95% 신뢰구간, Cohen's d, Welch t-test의 p를 구하고 Benjamini–Hochberg(BH)로 보정한다.

In [ ]:
import numpy as np, pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

rng = np.random.default_rng(7)
n, n_feat = 12, 200                      # 군당 12명, feature 200개
true_d = np.zeros(n_feat); true_d[:10] = 2.0             # 앞의 10개만 진짜 차이
ctrl = rng.normal(0, 1, (n, n_feat))
case = rng.normal(true_d, 1.5, (n, n_feat))               # 환자군은 분산도 더 큼

rows = []
for j in range(n_feat):
    a, b = ctrl[:, j], case[:, j]
    res = stats.ttest_ind(b, a, equal_var=False)          # Welch t-test
    ci = res.confidence_interval(0.95)                    # 평균 차이의 95% CI
    sd_pooled = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)   # 두 군 n이 같을 때
    rows.append((f"f{j + 1:03d}", true_d[j] != 0, b.mean() - a.mean(),
                 ci.low, ci.high, (b.mean() - a.mean()) / sd_pooled, res.pvalue))
out = pd.DataFrame(rows, columns=["feature", "진짜차이", "평균차", "CI_low", "CI_high", "cohen_d", "p"])
out["q_BH"] = multipletests(out["p"], alpha=0.05, method="fdr_bh")[1]

print(out.sort_values("p").head(5).round(4).to_string(index=False))
print(f"\n검정 {n_feat}개, 진짜 차이 {int(out.진짜차이.sum())}개")
print(f"p < 0.05: {(out.p < 0.05).sum()}개 (그중 진짜 {(out.p[out.진짜차이] < 0.05).sum()}개)")
print(f"q < 0.05: {(out.q_BH < 0.05).sum()}개 (그중 진짜 {(out.q_BH[out.진짜차이] < 0.05).sum()}개)")
assert (out.q_BH >= out.p).all()

### 예시 2 — 표본 수와 검정력

In [ ]:
from statsmodels.stats.power import TTestIndPower

power = TTestIndPower()
print("효과크기 d → 검정력 80%에 필요한 군당 표본 수 (양측 alpha 0.05)")
for d in [0.2, 0.5, 0.8, 1.2]:
    print(f"  d = {d}: {power.solve_power(effect_size=d, alpha=0.05, power=0.8):6.1f}명")
print("군당 10명일 때 검정력")
for d in [0.5, 0.8, 1.2]:
    print(f"  d = {d}: {power.power(effect_size=d, nobs1=10, alpha=0.05):.2f}")